In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'contador': int,
    'TIPOBITO': str,
    'IDADE': str,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': str,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': str,
    'CODMUNOCOR': str,
    'CAUSABAS': str     # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    'Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')


In [ ]:
# Mostra o tipo de dado das colunas convertidas em data real (formato: YYYY-mm-dd)
print(mortalidade[['DTOBITO', 'DTNASC']].dtypes)

# Mostra o formato das datas convertidas (YYYY-mm-dd)
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
# 1. Separar a IDADE em Tipo (1º dígito) e Quantidade (2º e 3º dígitos)
mortalidade['IDADELIMPA'] = ("000" + mortalidade['IDADE'].astype(str)).str[-3:]

tipo_idade = mortalidade['IDADELIMPA'].str[0]
qtde_idade = pd.to_numeric(mortalidade['IDADELIMPA'].str[1:], errors='coerce')

qtde_idade.loc[qtde_idade == 0] = 1

# 2. Aplicar a regra de conversão para ANOS
mortalidade['IDADEANOS'] = np.select(
    [
        tipo_idade == '0',  # Minutos
        tipo_idade == '1',  # Horas
        tipo_idade == '2',  # Dias
        tipo_idade == '3',  # Meses
        tipo_idade == '4',  # Anos (< 100)
        tipo_idade == '5'   # Anos (>= 100)
    ],
    [
        qtde_idade / (60 * 24 * 365),
        qtde_idade / (24 * 365),
        qtde_idade / 365,
        qtde_idade / 12,
        qtde_idade,
        qtde_idade + 100
    ],
    default=np.nan  # Idades ignoradas ou nulas
)

# 3. Remove a coluna temporária auxiliar
# mortalidade.drop(columns=['IDADELIMPA'], inplace=True)

mortalidade.head()

In [ ]:
# Usando o método .query() (Sintaxe mais limpa)
# mortalidade.query("IDADEANOS < 0.01")

# mortalidade[mortalidade['IDADEANOS'] <= 0.00274] # Idade < que 1 dia
mortalidade[mortalidade['IDADEANOS'] > 0.00274] # Idade > que 1 dia
# mortalidade[(mortalidade['IDADELIMPA'] == '200') & (mortalidade['CODMUNRES'] == '351360')]
# mortalidade[mortalidade['IDADELIMPA'] == '201']
# mortalidade[mortalidade['CODMUNRES'] == '351360']

In [ ]:
mortalidade.describe().round(7)

In [ ]:
mortalidade.info()

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'CODMUNRES',
    'MUNICIPIO',
    'UF',
    'POPULAÇÃO'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'CODMUNRES': str,
    'MUNICIPIO': str,
    'UF': str,
    'POPULAÇÃO': str
}

# 3. Importa o arquivo CSV de forma otimizada
municipios = pd.read_excel(
    'MUNICIPIOS.xlsx',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    na_values=['null']
)

In [ ]:
# Substitui os nulos da coluna por 0 e converte para inteiro comum
municipios['POPULAÇÃO'] = municipios['POPULAÇÃO'].fillna(0).astype('int64')

# Renomeando a coluna
municipios = municipios.rename(columns={'CODMUNRES': 'CODMUNOCOR'})

municipios.info()

In [ ]:
mort_muni = pd.merge(mortalidade, municipios, on='CODMUNOCOR', how='left')

mort_muni.head()